Transforms
Data does not always come in its final processed form that is required for training machine learning algorithms. We use transforms to perform some manipulation of the data and make it suitable for training.

All TorchVision datasets have two parameters -transform to modify the features and target_transform to modify the labels - that accept callables containing the transformation logic. The torchvision.transforms module offers several commonly-used transforms out of the box.

The FashionMNIST features are in PIL Image format, and the labels are integers. For training, we need the features as normalized tensors, and the labels as one-hot encoded tensors. To make these transformations, we use the torchvision.transforms.v2 API along with torch.nn.functional.one_hot.

In [1]:
import torch
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import v2

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    target_transform=v2.Lambda(
        lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
    ),
)

ToImage() and ToDtype()
The torchvision.transforms.v2 API replaces the legacy ToTensor transform with a two-step pipeline. v2.ToImage converts a PIL image or NumPy ndarray into a torchvision.tv_tensors.Image tensor, and v2.ToDtype with scale=True casts it to float32 and scales the pixel intensity values to the range [0., 1.].

Lambda Transforms
Lambda transforms apply any user-defined lambda function. Here, we use torch.nn.functional.one_hot to turn the integer label into a one-hot encoded tensor of size 10 (the number of labels in our dataset), then cast it to float to match the expected dtype.

In [2]:
target_transform = v2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
)

In [3]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4])


In [8]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [9]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [10]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
Non-linear activations are what create the complex mappings between the model's inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use nn.ReLU between our linear layers, but there's other activations to introduce non-linearity in your model.

In [11]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.1673, -0.2860,  0.0572, -0.2288, -0.0089,  0.5915,  0.4835, -0.5613,
         -0.0465, -0.2298,  0.2161, -0.0424, -0.2243, -0.0033, -0.1226,  0.1957,
         -0.3226, -0.1381, -0.1058,  0.4318],
        [ 0.1861,  0.4782,  0.0225, -0.1184, -0.1608,  0.4970,  0.3884, -0.1639,
         -0.0348, -0.0039,  0.6997, -0.0067,  0.1592,  0.2471,  0.0703,  0.2966,
         -0.2782, -0.0150,  0.3259,  0.3499],
        [ 0.1429, -0.0489,  0.3210, -0.4875, -0.0559,  0.3586,  0.4159, -0.2936,
          0.0868, -0.1389,  0.1053,  0.2716, -0.1176,  0.0680,  0.1107,  0.2054,
         -0.2697,  0.0964, -0.0471,  0.1487]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1673, 0.0000, 0.0572, 0.0000, 0.0000, 0.5915, 0.4835, 0.0000, 0.0000,
         0.0000, 0.2161, 0.0000, 0.0000, 0.0000, 0.0000, 0.1957, 0.0000, 0.0000,
         0.0000, 0.4318],
        [0.1861, 0.4782, 0.0225, 0.0000, 0.0000, 0.4970, 0.3884, 0.0000, 0.0000,
         0.0000, 0.6997, 0.0000, 0.1592, 0.2471, 0.07

nn.Sequential
nn.Sequential is an ordered container of modules. The data is passed through all the modules in the same order as defined. You can use sequential containers to put together a quick network like seq_modules.

In [12]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [13]:
print(f"Logits: {logits}\n\n")

Logits: tensor([[-0.2203, -0.2178, -0.0061,  0.2992, -0.1654, -0.1669, -0.3665,  0.2452,
         -0.2789, -0.0490],
        [-0.2507, -0.1618,  0.0912,  0.3125, -0.1342, -0.0857, -0.2041,  0.2662,
         -0.3596,  0.0890],
        [-0.1933, -0.1965,  0.0730,  0.3574,  0.0289, -0.1540, -0.2293,  0.2717,
         -0.2927, -0.0234]], grad_fn=<AddmmBackward0>)




nn.Softmax
The last linear layer of the neural network returns [logits]{.title-ref}

raw values in [-infty, infty] - which are passed to the nn.Softmax module. The logits are scaled to values [0, 1] representing the model's predicted probabilities for each class. dim parameter indicates the dimension along which the values must sum to 1.

In [14]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Model Parameters
Many layers inside a neural network are parameterized, i.e. have associated weights and biases that are optimized during training. Subclassing nn.Module automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model's parameters() or named_parameters() methods.

In this example, we iterate over each parameter, and print its size and a preview of its values.

In [15]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0098, -0.0011, -0.0155,  ...,  0.0228, -0.0113,  0.0168],
        [-0.0168,  0.0202, -0.0077,  ...,  0.0341,  0.0357,  0.0299]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0318, 0.0022], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0035,  0.0440,  0.0437,  ..., -0.0011,  0.0266, -0.0085],
        [ 0.0215,  0.0426, -0.0292,  ..., -0.0003, -0.0208, -0.0253]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si